In [51]:
import os
from pathlib import Path
from pydantic import BaseModel, Field
from pydantic_ai import Agent
from helpers.utils import get_prompt
import json
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [52]:
Path = r"D:\Kenpath\oan-evaluation"

In [53]:
os.chdir(Path)

In [54]:
from dotenv import load_dotenv
load_dotenv()


True

In [55]:
class Metric(BaseModel):
    metric_name: str = Field(..., description="The name of the metric to evaluate")
    score: int = Field(..., description="The score of the metric", ge=0, le=10)
    explanation: str = Field(..., description="The explanation of the score")

In [56]:
class EvaluationResult(BaseModel):
    """Result of the evaluation."""
    source_attribution: Metric = Field(..., description="Measures how well the response references appropriate, authoritative, and relevant sources. Higher scores reflect correct alignment between cited sources and the information presented.")
    grammatical_accuracy: Metric = Field(..., description="Evaluates the degree to which the response maintains grammatical correctness, valid word usage, accurate translations or transliterations, and proper sentence structure in Marathi.")
    factual_correctness: Metric = Field(..., description="Measures the accuracy of all factual information in the response, including details related to government schemes, mandi prices, fertilizers, pesticides, quantities, timelines, and agricultural guidelines, without introducing incorrect or misleading claims")
    actionability: Metric = Field(..., description="Evaluates the extent to which the response provides practical and implementable guidance, such as quantities, ratios, step-by-step instructions, timelines, or decision criteria that can be acted upon directly by the user.")
    completeness: Metric = Field(..., description="Measures how thoroughly the response addresses the user's query, including coverage of essential steps, conditions, data points, constraints, and safety considerations where relevant.")
    context_alignment: Metric = Field(..., description="Measures how closely the response stays aligned with the user's question and situational context, avoiding unnecessary or unrelated information, and introducing follow-up questions only when contextually justified.")
    information: Metric = Field(..., description="Evaluates whether the level of detail is suitable for the intended user, balancing clarity and precision without being overly technical or overly vague, and reflecting real-world agricultural practices.")
    safety_compliance: Metric = Field(..., description="Measures adherence to regulatory and safety norms by avoiding recommendations of banned, unsafe, or non-compliant agricultural inputs or practices, and by including appropriate safety precautions where applicable.")
    tool_usage: Metric = Field(..., description="Measures whether the response reflects correct selection and sequencing of tools or processes as required by the task, without omission, misuse, or unnecessary invocation.")
    retrieval_grounding: Metric = Field(..., description="Evaluates the extent to which the response is grounded in retrieved or provided documents and avoids introducing unsupported, speculative, or fabricated information.")

In [57]:
evaluation_agent = Agent(
    model='gpt-5',
    name="Evaluation Agent",
    instrument=True,
    output_type=EvaluationResult,
    retries=2,
    system_prompt=get_prompt('evaluation_system'),
)

Response-1

In [58]:
text = ["""User Query: मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?
Language: Marathi
==================================================

Assistant called tool: search_terms
Arguments: {"term":"मोसंबी","threshold":0.7}

Tool 'search_terms' returned:
Matching Terms for `मोसंबी`

Sweet Lime -> मोसंबी (mosambi) [100%]
Mosami -> मोसंबी (mosambi) [100%]
==================================================

Assistant called tool: search_terms
Arguments: {"term":"सिंचन वेळापत्रक","threshold":0.7}

Tool 'search_terms' returned:
Matching Terms for `सिंचन वेळापत्रक`

Irrigation Scheduling -> सिंचन वेळापत्रक (simchana velapatraka) [100%]
Irrigation scheduling tool -> सिंचन वेळापत्रक (simchana velapatraka) [100%]
Fertigation schedule -> खत सिंचन वेळापत्रक (khata simchana velapatraka) [91%]
Flood irrigation scheduling -> पूरसिंचनाचे वेळापत्रक (purasimchanache velapatraka) [83%]
Fertilizer Schedule -> खत वेळापत्रक (khata velapatraka) [74%]
==================================================

Assistant called tool: search_documents
Arguments: {"query":"Sweet Lime Irrigation Scheduling"}

Tool 'search_documents' returned:
> Search Results for `Sweet Lime Irrigation Scheduling`

**Dr. Panjabrao Deshmukh Krishi Vidyapeeth**
```
| **Sr.No.** | **Crop [पीक]** | **Year** | **January** | **February** | **March** | **April** | **May** | **June** | **July** | **August** | **September** | **October** | **November** | **December** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1   | Sweet Lime [मोसंबी] | 1   | \-  | \-  | \-  | \-  | \-  | 40  | 22  | 17  | 20  | 21  | 20  |...
==================================================
==================================================

Final Response:
मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक खालीलप्रमाणे आहे:

- पेरणी किंवा लागवडीनंतर झाड वाढीच्या टप्प्यानुसार सिंचनाचे प्रमाण वेगवेगळे ठरते.
- प्रत्येक महिन्यानुसार (झाडाच्या*वयाप्रमाणे) दरमहा एका झाडाला लागणारे पाणी (लिटरमध्ये):

| झाडाचे वय (वर्षे) | जानेवारी | फेब्रुवारी | मार्च | एप्रिल | मे | जून | जुलै | ऑगस्ट | सप्टेंबर | ऑक्टोबर | नोव्हेंबर | डिसेंबर |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | पाणी द्यायची गरज नाही | पाणी द्यायची गरज नाही | पाणी द्यायची गरज नाही | पाणी द्यायची गरज नाही | पाणी द्यायची गरज नाही | 40 | 22 | 17 | 20 | 21 | 20 | 18 |
| 2 | 25 | 31 | 47 | 66 | 80 | 52 | 29 | 22 | 26 | 27 | 26 | 23 |
| 3 | 31 | 39 | 60 | 83 | 102 | 65 | 36 | 28 | 33 | 36 | 32 | 29 |
| 4 | 41 | 52 | 81 | 113 | 138 | 76 | 42 | 32 | 38 | 42 | 37 | 33 |
| 5 | 46 | 59 | 91 | 126 | 154 | 85 | 46 | 36 | 42 | 47 | 42 | 37 |
| 6 | 48 | 61 | 95 | 131 | 160 | 88 | 48 | 37 | 44 | 49 | 43 | 39 |
| 7 | 52 | 66 | 102 | 142 | 173 | 96 | 52 | 40 | 48 | 53 | 47 | 42 |

- हंगामी पाऊस सुरू असल्यास, हवामान व मातीच्या नमीवरुन सिंचन करावे.
- ठिबक सिंचन वापरणे फायदेशीर ठरते. एक झाडाला दोन ठिबक नोजल (प्रत्येकी 8 लिटर/तास) बसवाव्यात.
- हलक्या मातीला वेगवेगळ्या टप्प्यांमध्ये पाणी द्यावे; उन्हाळ्यात सिंचन अंतर कमी ठेवावे (५-७ दिवसांनी), इतर हंगामात ८-१० दिवसांनी.
- झाडाच्या आजूबाजूला मल्चिंग करावे, यामुळे मातीतील ओलावा जास्त दिवस टिकतो.

**टीप:** अधिक अचूक सिंचन नियोजनासाठी ‘फुले सिंचन वेळापत्रक’ मोबाईल अॅप वापरण्याचा सल्ला आहे.

**स्रोत: डॉ. पंजाबराव देशमुख कृषी विद्यापीठ, अकोला (Dr. Panjabrao Deshmukh Krishi Vidyapeeth)**

आपल्या भागातील हवामान किंवा पाण्याच्या उपलब्धतेनुसार सविस्तर मार्गदर्शन हवे असल्यास कृपया सांगा!"""]

In [59]:
# import import_ipynb
from notebooks import *

In [60]:
eval_result = await evaluation_agent.run(text)

In [61]:
print(json.dumps(eval_result.output.model_dump(), indent=2, ensure_ascii=False))

{
  "source_attribution": {
    "metric_name": "source_attribution",
    "score": 6,
    "explanation": "The response cites Dr. Panjabrao Deshmukh Krishi Vidyapeeth (PDKV) and presents a month-by-month schedule that appears to align with the retrieved table snippet for Year 1 (e.g., June 40, July 22, August 17). However, it likely misinterprets hyphens in Jan–May as “no irrigation needed,” and it adds unsourced recommendations (e.g., a specific mobile app, drip emitter specs) that are not clearly attributed to the cited source."
  },
  "grammatical_accuracy": {
    "metric_name": "grammatical_accuracy",
    "score": 8,
    "explanation": "Marathi is generally clear and well-structured. Minor issues include a stray asterisk in “झाडाच्या*वयाप्रमाणे” and occasional formatting inconsistencies. Overall grammar and readability are good."
  },
  "factual_correctness": {
    "metric_name": "factual_correctness",
    "score": 5,
    "explanation": "There are potential inaccuracies/ambiguities: 

In [64]:
text_str = text[0] if isinstance(text, list) else text

query = text_str.split("User Query:")[1].splitlines()[0].strip()
language = text_str.split("Language:")[1].splitlines()[0].strip()
response = text_str.split("Final Response:")[1].strip()
df = pd.DataFrame.from_dict(eval_result.output.model_dump(), orient="index").reset_index(drop=True)
df.insert(0, "query", query)
df.insert(1, "response", response)
df.insert(2, "language", language)

In [65]:
df

,query,response,language,metric_name,score,explanation
0,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,source_attribution,6,The response cites Dr. Panjabrao Deshmukh Kris...
1,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,grammatical_accuracy,8,Marathi is generally clear and well-structured...
2,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,factual_correctness,5,There are potential inaccuracies/ambiguities: ...
3,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,actionability,7,The response provides a detailed schedule by m...
4,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,completeness,8,It covers an age-wise monthly schedule and add...
5,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,context_alignment,9,The response stays focused on the user’s reque...
6,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,information,6,"Information is detailed and relevant, but qual..."
7,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,safety_compliance,10,No unsafe or banned practices are recommended....
8,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,tool_usage,9,Appropriate tools (term matching and document ...
9,मोसंबी साठी आदर्श सिंचन वेळापत्रक काय आहे?,मोसंबी (Sweet Lime) साठी आदर्श सिंचन वेळापत्रक...,Marathi,retrieval_grounding,6,Core schedule seems to come from the retrieved...
